In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 📊 ETH OHLCV Data Exploration\n",
    "## Using SQLite Database\n",
    "\n",
    "This notebook explores Ethereum OHLCV (Open, High, Low, Close, Volume) data stored in SQLite database."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Import Libraries"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import sqlite3\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from pathlib import Path\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set style for better visualizations\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Connect to SQLite Database"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Define database path\n",
    "DATA_DIR = Path('..') / 'data'\n",
    "db_path = DATA_DIR / 'ETH.db'\n",
    "\n",
    "# Connect to SQLite database\n",
    "conn = sqlite3.connect(db_path)\n",
    "\n",
    "# Check if table exists\n",
    "tables = pd.read_sql(\"SELECT name FROM sqlite_master WHERE type='table';\", conn)\n",
    "print(\"📋 Tables in database:\")\n",
    "print(tables)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load data from SQLite\n",
    "query = \"SELECT * FROM eth_ohlcv ORDER BY date\"\n",
    "df = pd.read_sql(query, conn, parse_dates=['date'])\n",
    "\n",
    "# Close connection\n",
    "conn.close()\n",
    "\n",
    "print(f\"✅ Loaded {len(df)} days of OHLCV data from SQLite\")\n",
    "print(f\"📅 Period: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}\")\n",
    "df.head(10)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Basic Statistics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"📊 Basic Statistics:\")\n",
    "print(\"=\"*50)\n",
    "df[['open', 'high', 'low', 'close', 'volume']].describe()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Check for Missing Values"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"🔍 Missing Values:\")\n",
    "df.isnull().sum()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Correlation Matrix"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Correlation matrix\n",
    "correlation = df[['open', 'high', 'low', 'close', 'volume']].corr()\n",
    "\n",
    "plt.figure(figsize=(10, 8))\n",
    "sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True)\n",
    "plt.title('📈 Correlation Matrix - ETH Price Metrics', fontsize=14)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Price Distribution Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 2, figsize=(14, 10))\n",
    "\n",
    "# Close price distribution\n",
    "axes[0, 0].hist(df['close'], bins=50, edgecolor='black', alpha=0.7, color='blue')\n",
    "axes[0, 0].set_title('Close Price Distribution')\n",
    "axes[0, 0].set_xlabel('Price (USD)')\n",
    "axes[0, 0].set_ylabel('Frequency')\n",
    "axes[0, 0].axvline(df['close'].mean(), color='red', linestyle='--', \n",
    "                   label=f'Mean: ${df[\"close\"].mean():.2f}')\n",
    "axes[0, 0].legend()\n",
    "\n",
    "# Volume distribution\n",
    "axes[0, 1].hist(df['volume'], bins=50, edgecolor='black', alpha=0.7, color='green')\n",
    "axes[0, 1].set_title('Volume Distribution')\n",
    "axes[0, 1].set_xlabel('Volume')\n",
    "axes[0, 1].set_ylabel('Frequency')\n",
    "\n",
    "# Box plot for price\n",
    "axes[1, 0].boxplot(df['close'])\n",
    "axes[1, 0].set_title('Close Price Box Plot')\n",
    "axes[1, 0].set_ylabel('Price (USD)')\n",
    "\n",
    "# Box plot for volume\n",
    "axes[1, 1].boxplot(df['volume'])\n",
    "axes[1, 1].set_title('Volume Box Plot')\n",
    "axes[1, 1].set_ylabel('Volume')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Daily Returns Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Calculate returns\n",
    "df['returns'] = df['close'].pct_change() * 100\n",
    "df['log_returns'] = np.log(df['close'] / df['close'].shift(1)) * 100\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Returns over time\n",
    "axes[0].plot(df['date'], df['returns'], color='blue', alpha=0.7, linewidth=0.8)\n",
    "axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)\n",
    "axes[0].set_title('Daily Returns (%) Over Time')\n",
    "axes[0].set_xlabel('Date')\n",
    "axes[0].set_ylabel('Return (%)')\n",
    "axes[0].grid(True, alpha=0.3)\n",
    "\n",
    "# Returns distribution\n",
    "axes[1].hist(df['returns'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='purple')\n",
    "axes[1].axvline(df['returns'].mean(), color='red', linestyle='--', \n",
    "                label=f'Mean: {df[\"returns\"].mean():.2f}%')\n",
    "axes[1].axvline(df['returns'].median(), color='green', linestyle='--', \n",
    "                label=f'Median: {df[\"returns\"].median():.2f}%')\n",
    "axes[1].set_title('Returns Distribution')\n",
    "axes[1].set_xlabel('Return (%)')\n",
    "axes[1].set_ylabel('Frequency')\n",
    "axes[1].legend()\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(\"📊 Returns Statistics:\")\n",
    "print(f\"   Mean: {df['returns'].mean():.2f}%\")\n",
    "print(f\"   Std Dev: {df['returns'].std():.2f}%\")\n",
    "print(f\"   Skewness: {df['returns'].skew():.2f}\")\n",
    "print(f\"   Kurtosis: {df['returns'].kurtosis():.2f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. SQL Queries Examples"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Reconnect for SQL queries\n",
    "conn = sqlite3.connect(db_path)\n",
    "\n",
    "# Query: Top 10 highest price days\n",
    "top_days_query = \"\"\"\n",
    "SELECT date, open, high, low, close, volume\n",
    "FROM eth_ohlcv\n",
    "ORDER BY close DESC\n",
    "LIMIT 10\n",
    "\"\"\"\n",
    "top_days = pd.read_sql(top_days_query, conn)\n",
    "print(\"📈 Top 10 Highest Price Days:\")\n",
    "top_days"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Query: Monthly averages\n",
    "monthly_query = \"\"\"\n",
    "SELECT \n",
    "    strftime('%Y-%m', date) as month,\n",
    "    COUNT(*) as days,\n",
    "    ROUND(AVG(close), 2) as avg_price,\n",
    "    ROUND(MIN(close), 2) as min_price,\n",
    "    ROUND(MAX(close), 2) as max_price,\n",
    "    ROUND(AVG(volume), 0) as avg_volume\n",
    "FROM eth_ohlcv\n",
    "GROUP BY month\n",
    "ORDER BY month DESC\n",
    "LIMIT 12\n",
    "\"\"\"\n",
    "monthly = pd.read_sql(monthly_query, conn)\n",
    "print(\"\\n📅 Last 12 Months Averages:\")\n",
    "monthly\n",
    "\n",
    "conn.close()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Summary\n",
    "\n",
    "✅ Data loaded successfully from SQLite\n",
    "✅ Basic statistics calculated\n",
    "✅ Missing values checked\n",
    "✅ Correlation matrix created\n",
    "✅ Price distribution analyzed\n",
    "✅ Returns calculated and visualized\n",
    "✅ SQL queries executed"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}